# Forecasting walkthrough (one commodity)

Mirrors `run.py` for a single commodity. Place `all_agricultural_products_data.csv` in `../data/` first.

In [ ]:
import sys
sys.path.append('..')
import matplotlib.pyplot as plt
from src.data import load_futures, commodity_series, train_test_split
from src.features import adf_test, stl_decompose
from src.models import naive_forecast, arima_forecast, sarimax_forecast, lstm_forecast
from src.evaluate import metrics

In [ ]:
df = load_futures('../data/all_agricultural_products_data.csv')
series = commodity_series(df, 'Coffee')
series.plot(title='Coffee: monthly mean close price');

In [ ]:
print('ADF (level):', adf_test(series))
print('ADF (12-month seasonal difference):', adf_test(series.diff(12)))
stl_decompose(series).plot();

In [ ]:
train, test = train_test_split(series)
h = len(test)
forecasts = {
    'naive': naive_forecast(train, h),
    'ARIMA(1,1,1)': arima_forecast(train, h),
    'SARIMAX': sarimax_forecast(train, h)[0],
    'LSTM': lstm_forecast(train, h),
}
for name, pred in forecasts.items():
    print(name, {k: round(v, 2) for k, v in metrics(test, pred).items()})

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(train.index, train, label='train', color='black')
plt.plot(test.index, test, label='actual', color='gray')
for name, pred in forecasts.items():
    plt.plot(test.index, pred, '--', label=name)
plt.legend();